**Model Analysis and Evaluation**

In [ ]:
import numpy as np
import tensorflow as tf
import keras

In [ ]:
# Load your data
images = np.load("data/images.npy")  
labels = np.load("data/labels.npy")

# Check if the model is in SavedModel format
model_path = "model/"  

try:
    # Try loading as a standard Keras model
    model = keras.models.load_model(model_path)
    print("Loaded as a Keras model")
except ValueError:
    # If it fails, use TFSMLayer
    print("Loading using TFSMLayer (inference only)...")
    model = keras.layers.TFSMLayer(model_path, call_endpoint="serving_default")

    # Create a wrapper model
    inputs = keras.Input(shape=images.shape[1:])
    outputs = model(inputs)
    model = keras.Model(inputs, outputs)

# Print model summary (only if it's a Keras model)
if isinstance(model, keras.Model):
    print(model.summary())

# Test model predictions
# Get model predictions (first 5 images)
predictions = model(images[:5])

# Check what keys exist in the dictionary output
print("Output keys:", predictions.keys())

# Extract the correct output
output_key = list(predictions.keys())[0]  # Automatically selects the first key
predicted_values = predictions[output_key]  # Extract tensor

print("Sample predictions:", predicted_values.numpy())  # Convert to NumPy

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Convert predictions to class labels (if threshold-based classification)
predicted_labels = (predicted_values.numpy() > 0.5).astype(int)  # Example threshold

# Compute accuracy
accuracy = accuracy_score(labels[:5], predicted_labels) 
print("Accuracy:", accuracy)

# Print a detailed classification report
print(classification_report(labels[:5], predicted_labels))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Compute regression errors
mae = mean_absolute_error(labels[:5], predicted_values.numpy())
mse = mean_squared_error(labels[:5], predicted_values.numpy())

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")